In [ ]:
import os
import subprocess 

from dotenv import load_dotenv
import pandas as pd
import wandb
import matplotlib.pyplot as plt 
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
from mpl_toolkits.axes_grid1 import make_axes_locatable

load_dotenv()

name = os.getenv("WANDB_ENTITY")
project = os.getenv("WANDB_PROJECT")

api = wandb.Api()

DATA_ROOT = os.getenv("DATA_ROOT")

In [4]:
cols_to_keep = [
    'drop_vars_size', 'vf_total_params', 'test_probe_uncond_label_shape_acc',
    'test_probe_cond_value_scale_r2', 'test_probe_orig_value_x_position_r2',
    'test_probe_orig_value_scale_r2', 'hidden_dim',
    'test_probe_orig_value_orientation_sin_r2', 'test_probe_orig_value_orientation_cos_r2'
    'test_kl_loss', 'test_probe_uncond_value_scale_r2',
    'test_probe_cond_value_y_position_r2',
    'test_probe_orig_label_shape_acc', 'test_probe_cond_label_shape_acc',
    'n_layers', 'test_probe_cond_value_x_position_r2',
    'test_probe_uncond_value_y_position_r2', 'test_cfm_loss',
    'test_probe_uncond_value_orientation_sin_r2', 'test_probe_uncond_value_orientation_cos_r2',
    'test_probe_uncond_value_x_position_r2',
    'test_probe_cond_value_orientation_sin_r2', 'test_probe_cond_value_orientation_cos_r2', 'test_loss',
    'test_probe_orig_value_y_position_r2'
]

In [5]:
model_no = "7593802"
run_ids = [f"{model_no}_{i}" for i in range(27)]

config_keys = ['drop_vars_size', 'vf_total_params', 'hidden_dim', 'n_layers', '']

all_configs = []
all_val_histories = []

for rid in run_ids:
    run = api.run(f"/{name}/{project}/runs/{rid}")
    history = run.history(samples=100000)
    
    if history.empty:
        continue

    config_data = {'run_id': rid}
    for col in config_keys:
        if col in history.columns:
            series = history[col].dropna()
            config_data[col] = series.iloc[0] if not series.empty else None
            
    if config_data.get('n_layers') is not None and config_data.get('hidden_dim') is not None:
        config_data['config_label'] = f"L{int(config_data['n_layers'])}-H{int(config_data['hidden_dim'])}"
    
    
    all_configs.append(config_data)

    time_cols = [c for c in ['_step', 'epoch'] if c in history.columns]
    val_cols = [c for c in ['val_kl_loss', 'val_cfm_loss'] if c in history.columns]
    
    run_val_history = history[time_cols + val_cols].copy()
    
    run_val_history = run_val_history.dropna(subset=val_cols, how='all')
    
    run_val_history['run_id'] = rid
    
    all_val_histories.append(run_val_history)

master_df = pd.DataFrame(all_configs)
val_history_df = pd.concat(all_val_histories, ignore_index=True)

In [ ]:
plt.rcParams.update({'font.size': 16})
plt.rcParams.update({'axes.titlesize': 20})
plt.rcParams.update({'axes.labelsize': 18})

full_plot_df = val_history_df.merge(
    master_df[['run_id', 'vf_total_params', 'drop_vars_size']], 
    on='run_id'
)
full_plot_df = full_plot_df.sort_values(['run_id', 'epoch'])

label_map = {
    5.0: "Scale",
    3.0: "Scale + Position",
    1.0: "Scale + Position + Orientation"
}

unique_conds = sorted(full_plot_df['drop_vars_size'].unique(), reverse=True)
num_plots = len(unique_conds)

vmin, vmax = full_plot_df['vf_total_params'].min(), full_plot_df['vf_total_params'].max()
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
cmap = cm.viridis

# 2. Create the figure 
fig, axes = plt.subplots(2, num_plots, figsize=(6 * num_plots, 10), 
                         sharex=True, sharey='row', dpi=300)

if num_plots == 1:
    axes = axes.reshape(2, 1)

for i, cond_val in enumerate(unique_conds):
    cond_df = full_plot_df[full_plot_df['drop_vars_size'] == cond_val]
    ax_top = axes[0, i]
    ax_bottom = axes[1, i]

    for rid, group in cond_df.groupby('run_id'):
        param_val = group['vf_total_params'].iloc[0]
        color = cmap(norm(param_val))

        ax_top.plot(group['epoch'], group['val_cfm_loss'], 
                    color=color, linewidth=2.0, alpha=0.7, marker='o', markersize=3)
        ax_bottom.plot(group['epoch'], group['val_kl_loss'], 
                       color=color, linewidth=2.0, alpha=0.7, marker='o', markersize=3)

    # --- Formatting Both Rows ---
    for row_idx, ax in enumerate([ax_top, ax_bottom]):
        ax.set_yscale('log')
        ax.grid(True, which="major", linestyle='--', alpha=0.3)
        ax.grid(False, which="minor")
        
        # Ensure 3 ticks per axis
        ax.yaxis.set_major_locator(ticker.LogLocator(base=10.0, numticks=3))
        
        if row_idx == 0:
            ax.yaxis.set_major_formatter(ticker.ScalarFormatter())
            ax.yaxis.get_major_formatter().set_scientific(False)
        else:
            ax.yaxis.set_major_formatter(ticker.LogFormatterSciNotation())

    ax_top.set_title(label_map.get(cond_val, f"Cond: {cond_val}"), 
                     fontweight='bold', pad=12, fontsize=22)

    if i == 0:
        ax_top.set_ylabel(r'$\mathcal{L}_{\text{CFM}}$', labelpad=10, fontsize=24)
        ax_bottom.set_ylabel(r'$\text{KL} \left[ p(x_0|y)\| q(x_0) \right]$', labelpad=15, fontsize=24)

fig.supxlabel('Epoch', fontsize=26, fontweight='bold', y=0.04)


cax = fig.add_axes([0.91, 0.15, 0.02, 0.7]) 
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label('Total Parameters', labelpad=15, fontsize=22, fontweight='bold')

plt.subplots_adjust(
    left=0.08,   
    right=0.88,  
    bottom=0.15, 
    top=0.92, 
    wspace=0.1,  
    hspace=0.12  
)
plt.savefig("./val_loss_scale.pdf")
plt.show()

# Downstream inference 
## Scaling params

In [26]:
model_no = "7593802" 
all_results = []
run_ids = [f"{model_no}_{i}" for i in range(27)]

config_keys = ['drop_vars_size', 'vf_total_params', 'hidden_dim', 'n_layers']
keep_cols = config_keys + cols_to_keep

for rid in run_ids:
    run = api.run(f"/{name}/{project}/runs/{rid}")
    history = run.history(samples=100000)
    
    if history.empty:
        continue

    row_data = {'run_id': rid}
    
    for col in keep_cols:
        if col in history.columns:
            series = history[col].dropna()
            
            if series.empty:
                row_data[col] = None
            elif col in config_keys:
                row_data[col] = series.iloc[0]
            else:
                row_data[col] = series.iloc[-1]
        else:
            row_data[col] = run.summary.get(col)

    n_layers = row_data.get('n_layers')
    hidden_dim = row_data.get('hidden_dim')
    
    if n_layers is not None and hidden_dim is not None:
        row_data['config_label'] = f"L{int(n_layers)}-H{int(hidden_dim)}"
    
    all_results.append(row_data)

master_df = pd.DataFrame(all_results)

In [27]:
plots = ["test_probe_cond_value_scale_r2", "test_probe_cond_value_x_position_r2", "test_probe_cond_value_y_position_r2", "test_probe_cond_label_shape_acc"]

In [28]:
label_map = {
    5.0: "Scale Only",
    3.0: "Scale + Position",
    1.0: "Scale + Position + Orientation"
}

baseline_map = {
    'test_probe_cond_value_scale_r2': 'test_probe_orig_value_scale_r2',
    'test_probe_cond_value_x_position_r2': 'test_probe_orig_value_x_position_r2',
    'test_probe_cond_value_y_position_r2': 'test_probe_orig_value_y_position_r2',
    'test_probe_cond_value_orientation_r2': 'test_probe_orig_value_orientation_r2',
    'test_probe_cond_label_shape_acc': 'test_probe_orig_label_shape_acc',
}

uncond_map = {
    'test_probe_cond_value_scale_r2': 'test_probe_uncond_value_scale_r2',
    'test_probe_cond_value_x_position_r2': 'test_probe_uncond_value_x_position_r2',
    'test_probe_cond_value_y_position_r2': 'test_probe_uncond_value_y_position_r2',
    'test_probe_cond_value_orientation_r2': 'test_probe_uncond_value_orientation_r2',
    'test_probe_cond_label_shape_acc': 'test_probe_uncond_label_shape_acc',
}

# 2. Preparation: Sort by params and map labels
scaling_df = master_df.copy()
scaling_df['cond_label'] = scaling_df['drop_vars_size'].map(label_map)
scaling_df = scaling_df.sort_values('vf_total_params')

In [ ]:
# --- 0. Setup ---
plot_list = plots[:4] 
headings = ["Scale", "X Position", "Y Position", "Shape"] 
# Slightly reduced height to make it more horizontal/compact
fig, axes = plt.subplots(1, 4, figsize=(24, 6.5), sharex=True, dpi=300)

colors = {
    'Scale Only': '#440154', 
    'Scale + Position': '#21918c', 
    'Scale + Position + Orientation': '#fde725'
}

# 1. Main Plotting Loop
for i, test in enumerate(plot_list):
    ax = axes[i]
    if test not in scaling_df.columns:
        continue
        
    x_min, x_max = scaling_df['vf_total_params'].min(), scaling_df['vf_total_params'].max()
    
    # --- A. Baselines ---
    orig_col = baseline_map.get(test)
    if orig_col and orig_col in scaling_df.columns:
        o_vals = pd.to_numeric(scaling_df[orig_col], errors='coerce').dropna()
        if not o_vals.empty:
            o_mean = o_vals.mean()
            ax.axhline(o_mean, color='black', linestyle=':', linewidth=2.5, zorder=1, label="Orig. Baseline")
            ax.fill_between([x_min, x_max], o_mean - o_vals.std(), o_mean + o_vals.std(), 
                            color='black', alpha=0.1, zorder=0)

    uncond_col = uncond_map.get(test)
    if uncond_col and uncond_col in scaling_df.columns:
        u_vals = pd.to_numeric(scaling_df[uncond_col], errors='coerce').dropna()
        if not u_vals.empty:
            u_mean = u_vals.mean()
            ax.axhline(u_mean, color='crimson', linestyle='--', linewidth=2, zorder=1, label="Uncond. Avg")
            ax.fill_between([x_min, x_max], u_mean - u_vals.std(), u_mean + u_vals.std(), 
                            color='crimson', alpha=0.05, zorder=0)

    # --- B. Scaling Lines ---
    for label in ["Scale Only", "Scale + Position", "Scale + Position + Orientation"]:
        group = scaling_df[scaling_df['cond_label'] == label].dropna(subset=[test])
        if not group.empty:
            ax.plot(
                group['vf_total_params'], 
                group[test], 
                color=colors[label],
                marker='o',
                linewidth=3.5,
                markersize=8,
                alpha=0.9,
                zorder=3,
                label=label 
            )

    # --- C. Formatting ---
    ax.set_xscale('log')
    if 'loss' in test.lower():
        ax.set_yscale('log')

    ax.set_title(headings[i], fontweight='bold', pad=12, fontsize=24)

    # Y-Axis Labels
    if 'r2' in test.lower():
        y_label = r'$R^2$'
    elif 'acc' in test.lower() or 'accuracy' in test.lower():
        y_label = 'Accuracy'
    else:
        y_label = 'Metric Value'
    ax.set_ylabel(y_label, labelpad=5, fontsize=24)

    ax.grid(True, which="both", linestyle='--', alpha=0.4)


fig.supxlabel('Total Parameters', fontsize=26, fontweight='bold', y=0.08)

handles, labels = axes[0].get_legend_handles_labels()
# Moved closer to top of plots (bbox_to_anchor 1.02 instead of 1.08)
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.05),
           ncol=5, frameon=True, fontsize=18, facecolor='white')

plt.subplots_adjust(
    left=0.05,    
    right=0.98,   
    bottom=0.22, 
    top=0.88,     
    wspace=0.18 
)
plt.show()